In [1]:
import os
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

DATA_PATH = "../data/raw/European_Bank.csv"
OUT_PATH = "../data/processed/bank_clean.csv"
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)

In [2]:
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

(10000, 14)


,Year,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,2025,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2025,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,2025,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,2025,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,2025,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:
# drop column values
df = df.drop(columns=["Year","CustomerId","Surname"])
print(df.shape)

(10000, 11)


In [4]:
# customers with zero balance
df["ZeroBalance"] = (df["Balance"] == 0).astype(int)

# merge 3 and 4 products into "3+" since those groups are small compare to their own
df["ProductGroup"] = df["NumOfProducts"].map({1:"1",2:"2",3:"3+",4:"3+"})

print(df["ZeroBalance"].mean())
print(df["ProductGroup"].value_counts())

0.3617
ProductGroup
1     5084
2     4590
3+     326
Name: count, dtype: int64


In [21]:
# Age bands
df["AgeBand"] = pd.cut(
    df["Age"],
    bins = [17,29,39,49,59,120],
    labels = ["18-29","30-39","40-49","50-59","60+"],
)
# Tenure bands
df["TenureBand"] = pd.cut(
    df["Tenure"],
    bins = [-1,2,5,8,10],
    labels = ["0-2","3-5","6-8","9-10"],
)
# credit score bands
df["CreditBand"] = pd.cut(
    df["CreditScore"],
    bins = [299,579,669,739,850],
    labels = ["Poor","Fair","Good","Very Good+"],
)
# Balance Band
df["BalanceBand"] = "Zero"
nz = df["Balance"] > 0
df.loc[nz,"BalanceBand"] = pd.qcut(
     df.loc[nz, "Balance"], q=3, labels=["Low","Mid","High"]
).astype(str)
df["BalanceBand"] = pd.Categorical(
    df["BalanceBand"], categories = ["Zero","Low","Mid","High"], ordered = True
)

In [23]:
# checking for nulls values and final shape
print("nulls:", df.isnull().sum().sum())
print(df.shape)

nulls: 0
(10000, 17)


In [26]:
# check for every band has sensible counts or not 
for col in ["AgeBand","TenureBand","CreditBand","BalanceBand","ProductGroup"]:
    print(f"\n{col}")
    print(df[col].value_counts(dropna=False).sort_index())

# First look of churn rate in each band
for col in ["AgeBand","TenureBand","CreditBand","BalanceBand","ProductGroup"]:
    print(f"\nchurn rate by {col}")
    print(df.groupby(col, observed=True)["Exited"].mean().round(4))


AgeBand
AgeBand
18-29    1641
30-39    4346
40-49    2618
50-59     869
60+       526
Name: count, dtype: int64

TenureBand
TenureBand
0-2     2496
3-5     3010
6-8     3020
9-10    1474
Name: count, dtype: int64

CreditBand
CreditBand
Poor          2362
Fair          3331
Good          2428
Very Good+    1879
Name: count, dtype: int64

BalanceBand
BalanceBand
Zero    3617
Low     2128
Mid     2127
High    2128
Name: count, dtype: int64

ProductGroup
ProductGroup
1     5084
2     4590
3+     326
Name: count, dtype: int64

churn rate by AgeBand
AgeBand
18-29    0.0756
30-39    0.1088
40-49    0.3079
50-59    0.5604
60+      0.2795
Name: Exited, dtype: float64

churn rate by TenureBand
TenureBand
0-2     0.2115
3-5     0.2076
6-8     0.1887
9-10    0.2130
Name: Exited, dtype: float64

churn rate by CreditBand
CreditBand
Poor          0.2202
Fair          0.2056
Good          0.1862
Very Good+    0.2022
Name: Exited, dtype: float64

churn rate by BalanceBand
BalanceBand
Zero    0.1382
Lo

In [27]:
df.to_csv(OUT_PATH, index=False)

check = pd.read_csv(OUT_PATH)
print(check.shape, df.shape)
print(check.dtypes)

(10000, 17) (10000, 17)
CreditScore          int64
Geography           object
Gender              object
Age                  int64
Tenure               int64
Balance            float64
NumOfProducts        int64
HasCrCard            int64
IsActiveMember       int64
EstimatedSalary    float64
Exited               int64
ZeroBalance          int64
ProductGroup        object
AgeBand             object
TenureBand          object
CreditBand          object
BalanceBand         object
dtype: object


### Findings
- Dropped: Year (constant 2025), CustomerId (unique ID), Surname.
- Created: ZeroBalance, ProductGroup (1 / 2 / 3+), AgeBand, TenureBand, CreditBand, BalanceBand.
- Final shape: 10,000 rows x 17 columns. No nulls.
- Saved to data/processed/bank_clean.csv.
- First look at churn: strongest by AgeBand (peak 56% at 50-59) and ProductGroup (3+ = 86%, 2 = 7.6%); zero-balance customers churn least (13.8%); tenure and credit bands are nearly flat.